In [1]:
import pandas as pd
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple, Set

NA_TOKENS_DEFAULT = {"", "na", "nan", "none", "null"}

def csv_regression_diff(
    left_csv: str,
    right_csv: str,
    key_col: str,
    out_prefix: str = "diff",
    tol: float = 0.0,
    casefold_columns: bool = False,
    na_tokens: Optional[Set[str]] = None,
    na_equal: bool = True,           # if True, NA==NA is considered equal
    ignore_cols: Optional[List[str]] = None,  # exact names to ignore (after casefold if enabled)
) -> Dict[str, pd.DataFrame]:
    """
    Compare two CSVs by key and column names (feature names), independent of row/column order.

    Outputs:
      {out_prefix}_cell_changes.csv         — per-cell diffs (key, column, left, right)
      {out_prefix}_rows_only_left.csv       — rows only in left file
      {out_prefix}_rows_only_right.csv      — rows only in right file
      {out_prefix}_cols_only_left.csv       — columns only in left file
      {out_prefix}_cols_only_right.csv      — columns only in right file
      {out_prefix}_column_summary.csv       — count of diffs per shared column

    Notes:
      - Numeric cells are compared with absolute tolerance `tol` when both sides parse as numbers.
      - Strings are compared after strip(); NA tokens normalized per `na_tokens`.
      - NA==NA handled by `na_equal`.
    """
    na_tokens = set(map(str.lower, na_tokens or NA_TOKENS_DEFAULT))
    ignore_cols = set(ignore_cols or [])

    # 1) Load as strings (stable & safe)
    left  = pd.read_csv(left_csv, dtype=str, keep_default_na=False).copy()
    right = pd.read_csv(right_csv, dtype=str, keep_default_na=False).copy()

    # 2) Optional casefold/trim column names for robust matching
    def _normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
        new_cols = []
        for c in df.columns:
            cc = c.strip()
            if casefold_columns:
                cc = cc.casefold()
            new_cols.append(cc)
        df = df.copy()
        df.columns = new_cols
        return df

    left = _normalize_cols(left)
    right = _normalize_cols(right)
    key_norm = key_col.strip().casefold() if casefold_columns else key_col.strip()

    if key_norm not in left.columns or key_norm not in right.columns:
        raise KeyError(f"Key column '{key_col}' (normalized='{key_norm}') must exist in both files.")

    # 3) Deduplicate & index by key
    left  = left.drop_duplicates(subset=[key_norm], keep="first").set_index(key_norm, drop=False)
    right = right.drop_duplicates(subset=[key_norm], keep="first").set_index(key_norm, drop=False)

    # 4) Row set logic
    left_keys  = set(map(str, left.index))
    right_keys = set(map(str, right.index))
    only_left_keys   = sorted(left_keys - right_keys)
    only_right_keys  = sorted(right_keys - left_keys)
    common_keys      = sorted(left_keys & right_keys)

    only_left_df  = left.loc[only_left_keys]   if only_left_keys  else pd.DataFrame(columns=left.columns)
    only_right_df = right.loc[only_right_keys] if only_right_keys else pd.DataFrame(columns=right.columns)
    both_left  = left.loc[common_keys]
    both_right = right.loc[common_keys]

    # 5) Column set logic (exclude key + ignored)
    def _colset(df: pd.DataFrame) -> Set[str]:
        return {c for c in df.columns if c != key_norm}

    Lcols = _colset(both_left)
    Rcols = _colset(both_right)

    # Apply ignore list after casefolding (if enabled)
    if casefold_columns:
        ignore_cols = {c.casefold() for c in ignore_cols}
    shared_cols = sorted((Lcols & Rcols) - ignore_cols)
    only_left_cols = sorted((Lcols - Rcols) - ignore_cols)
    only_right_cols = sorted((Rcols - Lcols) - ignore_cols)

    cols_only_left_df = pd.DataFrame({"column": only_left_cols})
    cols_only_right_df = pd.DataFrame({"column": only_right_cols})

    # 6) Helpers for tolerant, NA-aware comparison
    def _normalize_series(s: pd.Series) -> pd.Series:
        # strip, lower for NA detection only
        raw = s.astype(str).str.strip()
        low = raw.str.casefold()
        is_na = low.isin(na_tokens)
        # convert recognized NA tokens to actual NA; keep others as raw
        out = raw.mask(is_na, pd.NA)
        return out

    def _equal_with_tol(sa: pd.Series, sb: pd.Series) -> pd.Series:
        a = _normalize_series(sa)
        b = _normalize_series(sb)

        # Case 1: both NA
        both_na = a.isna() & b.isna()
        eq = pd.Series(False, index=a.index)

        if na_equal:
            eq = both_na.copy()
        else:
            eq.loc[both_na] = False  # NA vs NA treated as different if na_equal=False

        # Try numeric compare where possible
        a_num = pd.to_numeric(a, errors="coerce")
        b_num = pd.to_numeric(b, errors="coerce")
        both_num = a_num.notna() & b_num.notna()

        if tol is not None and tol >= 0:
            eq_num = (a_num[both_num] - b_num[both_num]).abs() <= tol
            eq.loc[both_num] = eq_num.values
        else:
            # no tolerance: exact numeric equality
            eq_num = (a_num[both_num] == b_num[both_num])
            eq.loc[both_num] = eq_num.values

        # Fallback to string equality for the rest (non-NA, non-numeric)
        remaining = ~(both_na | both_num)
        if remaining.any():
            eq_str = a[remaining].astype(str) == b[remaining].astype(str)
            eq.loc[remaining] = eq_str.values

        return eq

    # 7) Cell-level differences across shared columns
    changes_rows: List[Dict[str, Any]] = []
    col_counts: Dict[str, int] = {}

    for col in shared_cols:
        eq = _equal_with_tol(both_left[col], both_right[col])
        diff_mask = ~eq
        if diff_mask.any():
            diff_idx = eq.index[diff_mask]
            col_counts[col] = len(diff_idx)
            for k in diff_idx:
                changes_rows.append({
                    key_norm: k,
                    "column": col,
                    "left":  both_left.at[k, col],
                    "right": both_right.at[k, col],
                })
        else:
            col_counts[col] = 0

    cell_changes = pd.DataFrame(changes_rows, columns=[key_norm, "column", "left", "right"])

    # 8) Column diff summary
    col_summary = (
        pd.DataFrame({"column": list(col_counts.keys()), "diff_count": list(col_counts.values())})
        .sort_values(["diff_count", "column"], ascending=[False, True])
        .reset_index(drop=True)
    )

    # 9) Save outputs
    out_p = Path(out_prefix)
    out_p.parent.mkdir(parents=True, exist_ok=True)

    cell_changes.to_csv(f"{out_prefix}_cell_changes.csv", index=False)
    only_left_df.to_csv(f"{out_prefix}_rows_only_left.csv", index=False)
    only_right_df.to_csv(f"{out_prefix}_rows_only_right.csv", index=False)
    cols_only_left_df.to_csv(f"{out_prefix}_cols_only_left.csv", index=False)
    cols_only_right_df.to_csv(f"{out_prefix}_cols_only_right.csv", index=False)
    col_summary.to_csv(f"{out_prefix}_column_summary.csv", index=False)

    print("Done.")
    print(f"  cell changes        : {len(cell_changes)}  -> {out_prefix}_cell_changes.csv")
    print(f"  rows only left      : {len(only_left_df)} -> {out_prefix}_rows_only_left.csv")
    print(f"  rows only right     : {len(only_right_df)} -> {out_prefix}_rows_only_right.csv")
    print(f"  cols only left      : {len(cols_only_left_df)} -> {out_prefix}_cols_only_left.csv")
    print(f"  cols only right     : {len(cols_only_right_df)} -> {out_prefix}_cols_only_right.csv")
    print(f"  per-column summary  : {len(col_summary)}  -> {out_prefix}_column_summary.csv")

    return {
        "cell_changes": cell_changes,
        "rows_only_left": only_left_df,
        "rows_only_right": only_right_df,
        "cols_only_left": cols_only_left_df,
        "cols_only_right": cols_only_right_df,
        "column_summary": col_summary,
    }


In [ ]:

def add_event_id(path_in, path_out):
    df = pd.read_csv(path_in, dtype=str, keep_default_na=False)
    df["event_id"] = (
        df["case_id"].astype(str)
        + " | "
        + df["task"].astype(str)
        + " | "
        + df["timestamp"].astype(str)
    )
    df.to_csv(path_out, index=False)

add_event_id("left.csv", "left_with_event_id.csv")
add_event_id("right.csv", "right_with_event_id.csv")

In [ ]:
csv_regression_diff(
    r"C:\Users\vikto\Documents\BC-ER-Application\notebooks\application_data_preprocessed2_old_script.csv",
    r"C:\Users\vikto\Documents\BC-ER-Application\notebooks\application_data_preprocessed2_new_script_time.csv",
    key_col="APPLICATION_ID",      # your ID column
    out_prefix="bcer_diff_updated_script/batch1", # files will be written with this prefix
    tol=1e-6,                      # numeric tolerance
    casefold_columns=True,         # robust to case differences in column names
    na_tokens={"", "NA", "NaN", "Null", "None"},
    na_equal=True,                 # treat NA vs NA as equal
    ignore_cols=["last_updated"],  # optional: columns to ignore
)

Done.
  cell changes        : 87516  -> bcer_diff_updated_script/batch1_cell_changes.csv
  rows only left      : 0 -> bcer_diff_updated_script/batch1_rows_only_left.csv
  rows only right     : 4 -> bcer_diff_updated_script/batch1_rows_only_right.csv
  cols only left      : 1 -> bcer_diff_updated_script/batch1_cols_only_left.csv
  cols only right     : 1 -> bcer_diff_updated_script/batch1_cols_only_right.csv
  per-column summary  : 60  -> bcer_diff_updated_script/batch1_column_summary.csv


{'cell_changes':       application_id             column        left                right
 0            6964408        active_time          18                 17.0
 1            8690751       after_ruling           0                    1
 2            8692212       after_ruling           0                    1
 3            8692447       after_ruling           0                    1
 4            8692754       after_ruling           0                    1
 ...              ...                ...         ...                  ...
 87511        9986489  xrrf_new_det_date  2023-01-06  2023-01-06 09:23:12
 87512        9986582  xrrf_new_det_date  2023-01-09  2023-01-09 12:45:25
 87513        9986675  xrrf_new_det_date  2023-01-09  2023-01-09 08:53:31
 87514        9992734  xrrf_new_det_date  2022-11-25  2022-11-25 08:42:08
 87515        9995992  xrrf_new_det_date  2022-08-11  2022-08-11 14:37:26
 
 [87516 rows x 4 columns],
 'rows_only_left': Empty DataFrame
 Columns: [unnamed: 0, fn consul